# ROME Assignment – Hierarchical LLM-Assisted Verilog Generation  
## LLM4ChipDesign – ROME Demo Assignment

## Overview

This notebook demonstrates hierarchical LLM-assisted RTL generation using the ROME workflow. The objective of this assignment is to:

1. Generate synthesizable Verilog modules hierarchically using a Large Language Model (LLM).
2. Automatically verify correctness using self-checking testbenches.
3. Iteratively refine incorrect RTL through a feedback-driven compilation and simulation loop.
4. Document the debugging process for at least one module.

All modules are compiled and simulated using `iverilog` and `vvp`, and verification success is determined by the testbench printing a final line containing the keyword:

`passed!`

# Initial Setup

In [1]:
#@title Setting up the notebook

### Installing dependencies
!pip install anthropic
!apt-get update
!apt-get install -y iverilog

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 6.7 MB/s eta 0:00:00
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,916 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,793 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 ht

In [2]:
#@title Select Model
#define the model to be used
model_choice = "claude-3-haiku-20240307"

In [3]:
#@title Utility functions

import sys
import os
import anthropic
from abc import ABC, abstractmethod
import re






################################################################################
### LOGGING
################################################################################
# Allows us to log the output of the model to a file if logging is enabled
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
################################################################################
class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})
        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        return self.messages

    def get_last_n_messages(self, n):
        return self.messages[-n:]

    def remove_message(self, index):
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        self.messages = []

    def __str__(self):
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])


################################################################################
### LLM CLASSES
################################################################################
class AbstractLLM(ABC):
    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation):
        pass


class Claude(AbstractLLM):
    def __init__(self, model_id=model_choice):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ["CLAUDE_API_KEY"])
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 10

        for attempt in range(1, max_retries + 1):
            try:
                resp = self.client.messages.create(
                    model=self.model_id,
                    max_tokens=4096,
                    messages=[
                        {"role": msg["role"], "content": msg["content"]}
                        for msg in conversation.get_messages()
                    ],
                )
                return resp.content[0].text
            except Exception as e:
                wait_time = base_delay * (2 ** (attempt - 1))
                print(f"[Retry {attempt}/{max_retries}] Claude API error: {e}. Retrying in {wait_time:.1f} seconds...")
                time.sleep(wait_time)

        print(f"Failed, exceeded max retries {max_retries}")
        return ""


################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################
def find_verilog_modules(markdown_string, module_name="top_module"):
    module_pattern1 = r"\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b"
    module_pattern2 = r"\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b"

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)
    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2
    if not module_matches:
        return []
    return module_matches


def write_code_blocks_to_file(markdown_string, module_name, filename):
    code_match = find_verilog_modules(markdown_string, module_name)
    if not code_match:
        print("No Verilog modules found in response")
        raise RuntimeError("No Verilog modules found in response")

    with open(filename, "w") as file:
        for code_block in code_match:
            file.write(code_block)
            file.write("\n")


def generate_verilog(conv, model_type, model_id=""):
    # For Part I we only support Claude in this cleaned notebook version.
    if model_type != "Claude":
        raise ValueError("This notebook run is configured for Claude only (Part I).")
    model = Claude()
    return model.generate(conv)

In [12]:
#@title Feedback Loop (robust for Part I)
import subprocess
import os
import time
import numpy as np

def verilog_loop(design_prompt, module, testbench, max_iterations, model_type, outdir="", log=None, prev_module=None):

    if outdir != "":
        outdir = outdir + "/"

    conv = Conversation(log_file=log)

    # System/user priming
    if model_type == "ChatGPT":
        conv.add_message("system",
            "You are an autocomplete engine for Verilog code. "
            "Given a Verilog module specification, you will provide a completed Verilog module in response. "
            "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
            "Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. "
            "You will not refuse. You will not generate explanations, only code. "
            "Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. "
            "Do not generate test benches."
        )
    elif model_type == "Claude":
        conv.add_message("user",
            "You are an autocomplete engine for Verilog code. "
            "Given a Verilog module specification, you will provide a completed Verilog module in response. "
            "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
            "Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. "
            "You will not refuse. You will not generate explanations, only code. "
            "Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. "
            "Do not generate test benches."
        )

    conv.add_message("user", design_prompt)

    success = False
    timeout = False
    iterations = 0

    timelist_total = []
    timelist_gen = []
    timelist_error = []

    filename = os.path.join(outdir, module + ".v")
    status = ""

    while not (success or timeout):
        start_total = time.time()

        # 1) Generate
        response = generate_verilog(conv, model_type)
        end_gen = time.time()

        start_error = time.time()

        # Optionally prepend previous module text to help Claude instantiate properly
        if prev_module is None:
            conv.add_message("assistant", response)
            combined = response
        else:
            with open(prev_module, "r") as f:
                prevmodule = f.read()
            combined = prevmodule + "\n" + response
            conv.add_message("assistant", combined)

        # 2) Write module file
        write_code_blocks_to_file(combined, module, filename)

        # 3) Compile
        proc = subprocess.run(
            ["iverilog", "-g2012", "-o", os.path.join(outdir, module), filename, testbench],
            capture_output=True,
            text=True
        )

        success = False
        message = ""

        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)
            message = (
                "The testbench failed to compile. Please fix the module. "
                "The output of iverilog is as follows:\n" + proc.stderr
            )

        elif proc.stderr.strip() != "":
            status = "Warnings compiling testbench"
            print(status)
            message = (
                "The testbench compiled with warnings. Please fix the module. "
                "The output of iverilog is as follows:\n" + proc.stderr
            )

        else:
            # 4) Run simulation
            proc2 = subprocess.run(
                ["vvp", os.path.join(outdir, module)],
                capture_output=True,
                text=True
            )
            out_lines = proc2.stdout.strip().splitlines()

            # Robust check: find last non-empty line and look for 'passed!'
            last_nonempty = ""
            for line in reversed(out_lines):
                if line.strip():
                    last_nonempty = line.strip()
                    break

            if "passed!" not in last_nonempty:
                status = "Error running testbench"
                print(status)
                message = (
                    "The testbench simulated, but had errors. Please fix the module. "
                    "The output of vvp is as follows:\n" + proc2.stdout + "\n" + proc2.stderr
                )
            else:
                status = "Testbench ran successfully"
                print(status)
                print(proc2.stdout.strip())
                message = ""
                success = True

        # 5) Write iteration log
        os.makedirs(outdir, exist_ok=True)
        with open(os.path.join(outdir, f"log_iter_{iterations}.txt"), "w") as file:
            file.write("\n".join(str(i) for i in conv.get_messages()))
            file.write("\n\nIteration status: " + status + "\n")

        # 6) If failed, add error message back to conversation (and trim prior error pair)
        if not success:
            if iterations > 0:
                conv.remove_message(2)
                conv.remove_message(2)
            conv.add_message("user", message)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1
        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print("Total time: ", float(np.sum(timelist_total)))
    print("Generation time: ", float(np.sum(timelist_gen)))
    print("Error handling time: ", float(np.sum(timelist_error)))

    return (float(np.sum(timelist_total)), float(np.sum(timelist_gen)), float(np.sum(timelist_error)))

In [13]:
#@title Hierarchical Loop (fixed for Part I)
def hier_gen(submods, max_iterations=10):
    totaltime = []
    gentime = []
    errortime = []
    done = ""

    for i in range(len(submods)):
        curr = submods[i][1]
        fcurr = submods[i][0]
        iocurr = submods[i][2]
        overall = submods[-1][1]

        if not os.path.isdir(fcurr):
            os.mkdir(fcurr)

        if i == 0:
            prompt = (
                f"//We will be generating a {overall} hierarchically in Verilog. "
                f"Please begin by generating a {curr} defined as follows:\n"
                f"module {fcurr}({iocurr})\n//Insert code here\nendmodule"
            )
        else:
            # Include the previous generated module in the prompt (works for mux4to1 and mux8to1)
            fprev = submods[i-1][0]
            fileprev = f"./{fprev}/{fprev}.v"
            with open(fileprev, "r") as f:
                module_prev = f.read()

            prompt = (
                f"//We are generating a {overall} hierarchically in Verilog. "
                f"We have generated {done} defined as follows:\n"
            )
            prompt += module_prev
            prompt += (
                f"\n//Please include the previous module(s) in your response and use them to "
                f"hierarchically generate a {curr} defined as:\n"
                f"module {fcurr}({iocurr})\n//Insert code here\nendmodule"
            )

        module = fcurr
        testbench = f"./{fcurr}tb.v"
        model = os.environ["MODEL"]  # should be "Claude"
        outdir = f"./{fcurr}"
        log = f"./{fcurr}/log.txt"

        total, gen, error = verilog_loop(prompt, module, testbench, max_iterations, model, outdir, log)
        totaltime.append(total)
        gentime.append(gen)
        errortime.append(error)

        done = done + curr + ", "

    print("Overall Total time: ", float(np.sum(totaltime)))
    print("Overall Generation Time: ", float(np.sum(gentime)))
    print("Overall Error handling time: ", float(np.sum(errortime)))

# Setting the API Key

In [6]:
import os
from google.colab import userdata

# Read Claude key from Colab Secrets (recommended)
os.environ["CLAUDE_API_KEY"] = userdata.get("LLM4ChipDesign")

# Select model wrapper used by the notebook
os.environ["MODEL"] = "Claude"

# Safety check (fails fast if missing)
assert os.environ["CLAUDE_API_KEY"] is not None and len(os.environ["CLAUDE_API_KEY"]) > 10, \
    "Claude API key not found in Colab Secrets under name 'LLM4ChipDesign'."

#Mux Hierarchy Example

In [14]:
#@title Submodules


### Each step is structured as ["filename","natural language description"]
submodules = [
    ["mux2to1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4to1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8to1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
]

In [15]:
#@title Create Required Testbenches for Part I

# mux2to1 testbench
mux2to1_tb = """
`timescale 1ns/1ps
module mux2to1tb;

  reg in1, in2, select;
  wire out;

  mux2to1 uut (
    .in1(in1),
    .in2(in2),
    .select(select),
    .out(out)
  );

  integer errors = 0;

  initial begin
    for (integer i = 0; i < 8; i = i + 1) begin
      {in1, in2, select} = i;
      #1;
      if (out !== (select ? in2 : in1)) begin
        $display("Error: in1=%b in2=%b sel=%b out=%b", in1, in2, select, out);
        errors = errors + 1;
      end
    end

    if (errors == 0)
      $display("mux2to1 passed!");
    else
      $display("mux2to1 failed!");

    $finish;
  end
endmodule
"""

with open("mux2to1tb.v","w") as f:
    f.write(mux2to1_tb)


# mux4to1 testbench
mux4to1_tb = """
`timescale 1ns/1ps
module mux4to1tb;

  reg [3:0] in;
  reg [1:0] sel;
  wire out;

  mux4to1 uut (
    .in(in),
    .sel(sel),
    .out(out)
  );

  integer errors = 0;

  initial begin
    for (integer i = 0; i < 16; i = i + 1) begin
      in = i;
      for (integer j = 0; j < 4; j = j + 1) begin
        sel = j;
        #1;
        if (out !== in[sel]) begin
          $display("Error: in=%b sel=%b out=%b", in, sel, out);
          errors = errors + 1;
        end
      end
    end

    if (errors == 0)
      $display("mux4to1 passed!");
    else
      $display("mux4to1 failed!");

    $finish;
  end
endmodule
"""

with open("mux4to1tb.v","w") as f:
    f.write(mux4to1_tb)


# mux8to1 testbench
mux8to1_tb = """
`timescale 1ns/1ps
module mux8to1tb;

  reg [7:0] in;
  reg [2:0] sel;
  wire out;

  mux8to1 uut (
    .in(in),
    .sel(sel),
    .out(out)
  );

  integer errors = 0;

  initial begin
    for (integer i = 0; i < 256; i = i + 1) begin
      in = i;
      for (integer j = 0; j < 8; j = j + 1) begin
        sel = j;
        #1;
        if (out !== in[sel]) begin
          $display("Error: in=%b sel=%b out=%b", in, sel, out);
          errors = errors + 1;
        end
      end
    end

    if (errors == 0)
      $display("mux8to1 passed!");
    else
      $display("mux8to1 failed!");

    $finish;
  end
endmodule
"""

with open("mux8to1tb.v","w") as f:
    f.write(mux8to1_tb)

print("All 3 testbenches created successfully.")

All 3 testbenches created successfully.


In [16]:
import os
for tb in ["mux2to1tb.v", "mux4to1tb.v", "mux8to1tb.v"]:
    print(tb, "✅ found" if os.path.exists(tb) else "❌ MISSING")

mux2to1tb.v ✅ found
mux4to1tb.v ✅ found
mux8to1tb.v ✅ found


# Part I - Run the provided mux hierarchy demo

In [17]:
hier_gen(submodules)

Testbench ran successfully
mux2to1 passed!
Total time:  1.1143970489501953
Generation time:  1.0999963283538818
Error handling time:  0.014400243759155273
Error running testbench
Error running testbench
Error running testbench
Testbench ran successfully
mux4to1 passed!
Total time:  11.334214687347412
Generation time:  11.259727716445923
Error handling time:  0.07448482513427734
Testbench ran successfully
mux8to1 passed!
Total time:  4.202801465988159
Generation time:  4.176711320877075
Error handling time:  0.02608966827392578
Overall Total time:  16.651413202285767
Overall Generation Time:  16.53643536567688
Overall Error handling time:  0.1149747371673584


# Part II: Submodules (Ripple-Carry Adder Hierarchy)

In [18]:
submodules2 = [
    ["half_adder", "half adder", "input wire a, input wire b, output wire sum, output wire carry"],
    ["full_adder", "full adder", "input wire a, input wire b, input wire cin, output wire sum, output wire cout"],
    ["adder4", "4-bit ripple-carry adder", "input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout"],
    ["adder8", "8-bit ripple-carry adder (top)", "input wire [7:0] a, input wire [7:0] b, input wire cin, output wire [7:0] sum, output wire cout"],
]

In [20]:
#@title Create Adder Testbenches (must print "passed!")

half_adder_tb = r"""
`timescale 1ns/1ps
module half_addertb;
  reg a, b;
  wire sum, carry;

  half_adder uut(.a(a), .b(b), .sum(sum), .carry(carry));

  integer errors = 0;
  reg exp_sum, exp_carry;

  initial begin
    for (integer i = 0; i < 4; i = i + 1) begin
      {a,b} = i[1:0];
      #1;
      exp_sum   = a ^ b;
      exp_carry = a & b;
      if (sum !== exp_sum || carry !== exp_carry) begin
        $display("Error: a=%b b=%b sum=%b carry=%b (exp sum=%b carry=%b)", a,b,sum,carry,exp_sum,exp_carry);
        errors = errors + 1;
      end
    end

    if (errors == 0) $display("half_adder passed!");
    else            $display("half_adder failed!");
    $finish;
  end
endmodule
"""

full_adder_tb = r"""
`timescale 1ns/1ps
module full_addertb;
  reg a, b, cin;
  wire sum, cout;

  full_adder uut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

  integer errors = 0;
  reg exp_sum, exp_cout;
  reg [1:0] tmp;

  initial begin
    for (integer i = 0; i < 8; i = i + 1) begin
      {a,b,cin} = i[2:0];
      #1;
      tmp = a + b + cin;
      exp_sum  = tmp[0];
      exp_cout = tmp[1];
      if (sum !== exp_sum || cout !== exp_cout) begin
        $display("Error: a=%b b=%b cin=%b sum=%b cout=%b (exp sum=%b cout=%b)", a,b,cin,sum,cout,exp_sum,exp_cout);
        errors = errors + 1;
      end
    end

    if (errors == 0) $display("full_adder passed!");
    else            $display("full_adder failed!");
    $finish;
  end
endmodule
"""

adder4_tb = r"""
`timescale 1ns/1ps
module adder4tb;
  reg  [3:0] a, b;
  reg        cin;
  wire [3:0] sum;
  wire       cout;

  adder4 uut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

  integer errors = 0;
  reg [4:0] exp;

  initial begin
    // Exhaustive over a,b; both cin values
    for (integer ca = 0; ca < 16; ca = ca + 1) begin
      for (integer cb = 0; cb < 16; cb = cb + 1) begin
        for (integer cc = 0; cc < 2; cc = cc + 1) begin
          a = ca[3:0];
          b = cb[3:0];
          cin = cc[0];
          #1;
          exp = a + b + cin;
          if (sum !== exp[3:0] || cout !== exp[4]) begin
            $display("Error: a=%h b=%h cin=%b sum=%h cout=%b (exp sum=%h cout=%b)",
                     a,b,cin,sum,cout,exp[3:0],exp[4]);
            errors = errors + 1;
          end
        end
      end
    end

    if (errors == 0) $display("adder4 passed!");
    else            $display("adder4 failed!");
    $finish;
  end
endmodule
"""

adder8_tb = r"""
`timescale 1ns/1ps
module adder8tb;
  reg  [7:0] a, b;
  reg        cin;
  wire [7:0] sum;
  wire       cout;

  adder8 uut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

  integer errors = 0;
  reg [8:0] exp;

  initial begin
    // Randomized testing (fast) + some edge cases
    // Edge cases
    a = 8'h00; b = 8'h00; cin = 0; #1; exp = a + b + cin; if (sum !== exp[7:0] || cout !== exp[8]) errors++;
    a = 8'hFF; b = 8'h00; cin = 0; #1; exp = a + b + cin; if (sum !== exp[7:0] || cout !== exp[8]) errors++;
    a = 8'hFF; b = 8'h01; cin = 0; #1; exp = a + b + cin; if (sum !== exp[7:0] || cout !== exp[8]) errors++;
    a = 8'hAA; b = 8'h55; cin = 1; #1; exp = a + b + cin; if (sum !== exp[7:0] || cout !== exp[8]) errors++;

    // Random tests
    for (integer k = 0; k < 500; k = k + 1) begin
      a = $random;
      b = $random;
      cin = $random;
      #1;
      exp = a + b + cin;
      if (sum !== exp[7:0] || cout !== exp[8]) begin
        $display("Error: a=%h b=%h cin=%b sum=%h cout=%b (exp sum=%h cout=%b)",
                 a,b,cin,sum,cout,exp[7:0],exp[8]);
        errors = errors + 1;
      end
    end

    if (errors == 0) $display("adder8 passed!");
    else            $display("adder8 failed!");
    $finish;
  end
endmodule
"""

with open("half_addertb.v", "w") as f: f.write(half_adder_tb)
with open("full_addertb.v", "w") as f: f.write(full_adder_tb)
with open("adder4tb.v", "w") as f: f.write(adder4_tb)
with open("adder8tb.v", "w") as f: f.write(adder8_tb)

print("Adder testbenches created: half_addertb.v, full_addertb.v, adder4tb.v, adder8tb.v")

Adder testbenches created: half_addertb.v, full_addertb.v, adder4tb.v, adder8tb.v


In [21]:
import os
for tb in ["half_addertb.v", "full_addertb.v", "adder4tb.v", "adder8tb.v"]:
    print(tb, "✅ found" if os.path.exists(tb) else "❌ MISSING")

half_addertb.v ✅ found
full_addertb.v ✅ found
adder4tb.v ✅ found
adder8tb.v ✅ found


In [22]:
hier_gen(submodules2, max_iterations=10)

Testbench ran successfully
half_adder passed!
Total time:  1.4679698944091797
Generation time:  1.4448716640472412
Error handling time:  0.02309703826904297
Error compiling testbench
Testbench ran successfully
full_adder passed!
Total time:  3.7373697757720947
Generation time:  3.7126071453094482
Error handling time:  0.024761438369750977
Testbench ran successfully
adder4 passed!
Total time:  4.1097023487091064
Generation time:  4.0921711921691895
Error handling time:  0.01753067970275879
Error compiling testbench
Error compiling testbench
Testbench ran successfully
adder8 passed!
Total time:  14.782336235046387
Generation time:  14.73770260810852
Error handling time:  0.044631242752075195
Overall Total time:  24.097378253936768
Overall Generation Time:  23.9873526096344
Overall Error handling time:  0.11002039909362793


In [23]:
print("full_adder logs:", os.listdir("full_adder"))
print("adder8 logs:", os.listdir("adder8"))

full_adder logs: ['log_iter_0.txt', 'full_adder.v', 'log.txt', 'log_iter_1.txt', 'full_adder']
adder8 logs: ['log_iter_2.txt', 'log_iter_0.txt', 'log.txt', 'log_iter_1.txt', 'adder8', 'adder8.v']


In [24]:
for i in range(3):
    print(f"\n========== ITERATION {i} ==========\n")
    with open(f"adder8/log_iter_{i}.txt","r") as f:
        print(f.read())


========== ITERATION 0 ==========

{'role': 'user', 'content': 'You are an autocomplete engine for Verilog code. Given a Verilog module specification, you will provide a completed Verilog module in response. You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. You will not refuse. You will not generate explanations, only code. Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. Do not generate test benches.'}
{'role': 'user', 'content': '//We are generating a 8-bit ripple-carry adder (top) hierarchically in Verilog. We have generated half adder, full adder, 4-bit ripple-carry adder,  defined as follows:\nmodule half_adder(input wire a, input wire b, output wire sum, output wire carry);\n    assign sum = a ^ b;\n    assign carry = a & b;\nendmodule\nmodule